# 此文档用于实现更高级的网络结构
* CNN,RNN以及Transformer
## 1.CNN的实现
* 对于卷积的理解
> 信号分析中，卷积用于求冲激响应的累计效果
> 概率论中，卷积用于求线密度（想象一下一条线平移过平面，故命名为卷积）
> 图像处理中，卷积用于求卷积算子和图像数据点积（卷积算子滑过图像的过程类似于上面线滑过平面的过程
* 1*1的卷积核主要用于将变换图像的通道数

In [33]:
import torch as T

# 1*1卷积核实现
def corr2d_one(X,K):
    c_i,h,w = X.shape
    c_o = K.shape[0]
    X = X.reshape(c_i,h*w) # 相当于Flatten
    K = K.reshape(c_o,c_i)
    Y = T.matmul(K,X)
    return Y.reshape(c_o,h,w)

X = T.normal(0,1,size=(3,3,3))
K = T.normal(0,1,size=(2,3,1,1))

print(corr2d_one(X,K))

tensor([[[-0.1143,  0.4737,  0.0367],
         [ 0.2748,  0.2931,  0.3290],
         [-0.5537,  0.4485, -0.3726]],

        [[-0.2469,  1.4514,  1.5542],
         [-0.5878,  2.3781, -1.9455],
         [ 0.7309,  2.8021, -3.5875]]])


* Lenet实现

In [2]:
import torch as T
import torchvision as tv
import torch.nn as nn
import torch.nn.init as init

from torch.utils import data
from torchvision import transforms

# 检查是否有可用的 GPU
device = T.device('cuda' if T.cuda.is_available() else 'cpu')

# 定义数据生成器
def load_data(batch_size):
    # 将图像从 PIL 格式或 NumPy 数组转换为 PyTorch 的张量（tensor）
    trans = [transforms.ToTensor()]
    # 是一个用于将多个转换操作组合在一起的工具
    trans = transforms.Compose(trans)
    # 加载mnist数据集
    train_data = tv.datasets.MNIST(root='./data', train=True, download=True, transform=trans)
    test_data = tv.datasets.MNIST(root='./data', train=False, download=True, transform=trans)
    # 定义数据加载器
    train_iter = data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_iter = data.DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_iter, test_iter

# 定义网络并移动到 GPU
net = nn.Sequential(
    nn.Conv2d(1,6,kernel_size=(5,5),padding=(2,2)),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=(2,2),stride=(2,2)),
    nn.Conv2d(6,16,kernel_size=(5,5)),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=(2,2),stride=(2,2)),
    nn.Flatten(),
    nn.Linear(16*5*5,120),
    nn.ReLU(),
    nn.Linear(120,84),
    nn.ReLU(),
    nn.Linear(84,10)
).to(device)

# 初始化函数
def initialize_weights(model):
    for layer in model:
        if isinstance(layer, nn.Linear) or isinstance(layer, nn.Conv2d):
            init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # 使用 He 初始化
            if layer.bias is not None:
                init.zeros_(layer.bias)  # 偏置初始化为 0

# 调用初始化函数
initialize_weights(net)

# 损失函数
loss = nn.CrossEntropyLoss()

# 优化器
sgd = T.optim.SGD(net.parameters(), lr=0.1)

# 训练
epoch = 10
batch_size = 256
train_iter, test_iter = load_data(batch_size)
for i in range(epoch):
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)  # 将数据移动到 GPU
        sgd.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.backward()
        sgd.step()
    
    # 测试阶段
    net.eval()  # 设置模型为评估模式
    correct = 0
    total = 0
    test_loss = 0.0

    with T.no_grad():
        for X, y in test_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            test_loss += loss(y_hat, y).item()  # 累加测试损失
            predicted = T.argmax(y_hat, dim=1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    accuracy = correct / total
    print(f"Epoch: {i + 1}, Loss: {test_loss / len(test_iter):.4f}, Test Accuracy: {accuracy:.4f}")


Epoch: 1, Loss: 0.2811, Test Accuracy: 0.9085
Epoch: 2, Loss: 0.1123, Test Accuracy: 0.9652
Epoch: 3, Loss: 0.0916, Test Accuracy: 0.9696
Epoch: 4, Loss: 0.1141, Test Accuracy: 0.9646
Epoch: 5, Loss: 0.0652, Test Accuracy: 0.9786
Epoch: 6, Loss: 0.0629, Test Accuracy: 0.9782
Epoch: 7, Loss: 0.0601, Test Accuracy: 0.9797
Epoch: 8, Loss: 0.0713, Test Accuracy: 0.9762
Epoch: 9, Loss: 0.0641, Test Accuracy: 0.9790
Epoch: 10, Loss: 0.0481, Test Accuracy: 0.9850


* 2012年AlexNet横空出世
> 下面展示如何将图片的像素大小从28 * 28调整为224 * 224（底层原理是插值法）

In [2]:
from torch.utils import data
from torchvision import transforms
import torchvision as tv

# 定义数据生成器
def load_data(batch_size,resize):
    trans = [transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))]  # 将图像转换为张量,并归一化
    if resize:
        trans.insert(0,transforms.Resize(resize)) # 增广
    trans = transforms.Compose(trans) # 拼接所有操作
    # 加载mnist数据集
    train_data = tv.datasets.MNIST(root='./data', train=True, download=True, transform=trans)
    test_data = tv.datasets.MNIST(root='./data', train=False, download=True, transform=trans)
    # 定义数据加载器
    train_iter = data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_iter = data.DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_iter, test_iter

#train_iter, test_iter = load_data(128,224)
#for X, y in train_iter:
    #print(X.shape)
    #break

> AlexNet的结构(自定义实现)
> 等同于：
```python
net = nn.Sequential(
    nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(96, 256, kernel_size=5, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(256, 384, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(384, 384, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(384, 256, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Flatten(),
    nn.Linear(6400, 4096),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(4096, 4096),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(4096, 10),
)
```

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class AlexNet(nn.Module):
    def __init__(self):
        super(AlexNet, self).__init__()
        
        # 卷积层
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=96, kernel_size=11, stride=4)
        self.conv2 = nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, padding=1)

        # 最大池化层
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2)

        # 全连接层
        self.fc1 = nn.Linear(6400, 4096)
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, 10)

    def forward(self, x):
        # 卷积层 + ReLU + 最大池化
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(F.relu(self.conv5(x)))

        # 展平
        x = x.view(x.size(0), -1)  # 将多维输入展平为一维
        
        # 全连接层 + ReLU
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.5)  # dropout
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=0.5)  # dropout
        x = self.fc3(x)  # 输出层

        return x

> 初始化函数

In [4]:
import torch.nn.init as init

# 初始化函数
def initialize_weights(model):
    for layer in model.children(): # 这里因为net使用Class创建，所以使用children()，而使用sequential创建的模型可以直接遍历
        if isinstance(layer, nn.Linear) or isinstance(layer, nn.Conv2d):
            init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # 使用 He 初始化
            if layer.bias is not None:
                init.zeros_(layer.bias)  # 偏置初始化为 0
        elif isinstance(layer, nn.BatchNorm2d):
            init.ones_(layer.weight)  # 批归一化层的权重初始化为 1
            init.zeros_(layer.bias)  # 偏置初始化为 0

net = AlexNet()
initialize_weights(net)
# 打印模型的所有参数
#for name, param in net.named_parameters():
    #print(name, param.shape)
#print([layer for layer in net.children()])


* 训练

In [7]:
import torch as T
# 训练
epoch = 10
batch_size = 256
# 损失函数
loss = nn.CrossEntropyLoss()
# 优化器
optimizer = T.optim.SGD(net.parameters(), lr=0.01)

# 检查是否有可用的 GPU
device = T.device('cuda' if T.cuda.is_available() else 'cpu')
net.to(device)

train_iter, test_iter = load_data(batch_size,224)

for i in range(epoch):
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)  # 将数据移动到 GPU
        optimizer.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.backward()
        optimizer.step()
    
    # 测试阶段
    net.eval()  # 设置模型为评估模式
    correct = 0
    total = 0
    test_loss = 0.0

    with T.no_grad():
        for X, y in test_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            test_loss += loss(y_hat, y).item()  # 累加测试损失
            predicted = T.argmax(y_hat, dim=1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    accuracy = correct / total
    print(f"Epoch: {i + 1}, Loss: {test_loss / len(test_iter):.4f}, Test Accuracy: {accuracy:.4f}")

Epoch: 1, Loss: 0.0303, Test Accuracy: 0.9899
Epoch: 2, Loss: 0.0258, Test Accuracy: 0.9909
Epoch: 3, Loss: 0.0215, Test Accuracy: 0.9930
Epoch: 4, Loss: 0.0328, Test Accuracy: 0.9887
Epoch: 5, Loss: 0.0284, Test Accuracy: 0.9910
Epoch: 6, Loss: 0.0235, Test Accuracy: 0.9929
Epoch: 7, Loss: 0.0217, Test Accuracy: 0.9932
Epoch: 8, Loss: 0.0229, Test Accuracy: 0.9918
Epoch: 9, Loss: 0.0210, Test Accuracy: 0.9932
Epoch: 10, Loss: 0.0238, Test Accuracy: 0.9928


* 批量规范化
> 训练深层神经网络是十分困难的，特别是在较短的时间内使他们收敛更加棘手
> 批量规范化使得研究人员能够训练100层以上的网络

* Resnet（残差网）

## 2.RNN